# Genetic Algorithm Optimization

This notebook demonstrates genetic algorithm optimization for strategy parameters.

**Note**: This uses a simple genetic algorithm implementation. For production use,
consider libraries like DEAP or PyGAD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import kimsfinance_core
from kimsfinance.strategies import RSIStrategy

print("Genetic Algorithm Optimization Demo")
print("This is a simplified example - use specialized libraries in production")

## 1. Generate Data

In [ ]:
def generate_sample_data(n=2000, seed=42):
    np.random.seed(seed)
    timestamps = np.arange(n, dtype=np.int64) * 60
    base = np.linspace(100.0, 180.0, n)
    noise = np.random.randn(n).cumsum() * 3
    close = base + noise
    open_prices = close + np.random.randn(n) * 0.5
    high = np.maximum(open_prices, close) + np.abs(np.random.randn(n) * 2)
    low = np.minimum(open_prices, close) - np.abs(np.random.randn(n) * 2)
    volume = np.random.uniform(1000, 10000, n)
    return timestamps, open_prices, high, low, close, volume

timestamps, open_p, high, low, close, volume = generate_sample_data(2000)
print(f"Generated {len(close)} candles")

## 2. Simple Genetic Algorithm Implementation

In [ ]:
class GeneticOptimizer:
    def __init__(self, population_size=50, generations=20, mutation_rate=0.1):
        self.population_size = population_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.history = []
    
    def create_individual(self):
        """Create random RSI strategy parameters"""
        return {
            'period': np.random.choice([7, 10, 14, 20, 30]),
            'buy_threshold': np.random.uniform(20, 40),
            'sell_threshold': np.random.uniform(60, 80)
        }
    
    def evaluate_fitness(self, individual):
        """Evaluate strategy with given parameters"""
        strategy = RSIStrategy(
            period=int(individual['period']),
            buy_threshold=individual['buy_threshold'],
            sell_threshold=individual['sell_threshold']
        )
        
        try:
            result = kimsfinance_core.run_backtest(
                high=high, low=low, close=close, open_prices=open_p,
                volume=volume, timestamps=timestamps, strategy=strategy,
                initial_capital=10000.0, trading_fee=0.001,
                slippage=0.0005, use_gpu=False
            )
            # Fitness = Sharpe ratio (maximize)
            return result['sharpe_ratio']
        except:
            return -999  # Invalid strategy
    
    def crossover(self, parent1, parent2):
        """Single-point crossover"""
        child = {}
        for key in parent1:
            child[key] = parent1[key] if np.random.rand() < 0.5 else parent2[key]
        return child
    
    def mutate(self, individual):
        """Random mutation"""
        if np.random.rand() < self.mutation_rate:
            if np.random.rand() < 0.33:
                individual['period'] = np.random.choice([7, 10, 14, 20, 30])
            elif np.random.rand() < 0.5:
                individual['buy_threshold'] = np.random.uniform(20, 40)
            else:
                individual['sell_threshold'] = np.random.uniform(60, 80)
        return individual
    
    def optimize(self):
        """Run genetic algorithm"""
        # Initialize population
        population = [self.create_individual() for _ in range(self.population_size)]
        
        for gen in range(self.generations):
            # Evaluate fitness
            fitness = [self.evaluate_fitness(ind) for ind in population]
            
            # Record best
            best_idx = np.argmax(fitness)
            best_fitness = fitness[best_idx]
            best_individual = population[best_idx]
            self.history.append((best_fitness, best_individual.copy()))
            
            print(f"Generation {gen+1}/{self.generations} - Best Sharpe: {best_fitness:.3f} - "
                  f"Params: {best_individual}")
            
            # Selection (tournament)
            selected = []
            for _ in range(self.population_size):
                tournament = np.random.choice(len(population), size=3, replace=False)
                winner = tournament[np.argmax([fitness[i] for i in tournament])]
                selected.append(population[winner])
            
            # Crossover and mutation
            next_generation = []
            for i in range(0, self.population_size, 2):
                parent1 = selected[i]
                parent2 = selected[(i+1) % self.population_size]
                child1 = self.mutate(self.crossover(parent1, parent2))
                child2 = self.mutate(self.crossover(parent2, parent1))
                next_generation.extend([child1, child2])
            
            population = next_generation[:self.population_size]
        
        return self.history[-1][1]  # Return best individual

print("Genetic optimizer initialized")

## 3. Run Optimization

In [ ]:
optimizer = GeneticOptimizer(population_size=30, generations=10, mutation_rate=0.15)
best_params = optimizer.optimize()

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)
print(f"Best parameters found:")
print(f"  Period: {best_params['period']}")
print(f"  Buy threshold: {best_params['buy_threshold']:.2f}")
print(f"  Sell threshold: {best_params['sell_threshold']:.2f}")
print("="*60)

## 4. Visualize Evolution

In [ ]:
generations = range(1, len(optimizer.history) + 1)
best_fitness = [h[0] for h in optimizer.history]

plt.figure(figsize=(12, 6))
plt.plot(generations, best_fitness, marker='o', linewidth=2, markersize=6)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Best Sharpe Ratio', fontsize=12)
plt.title('Genetic Algorithm Evolution', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Test Optimized Strategy

In [ ]:
optimal_strategy = RSIStrategy(
    period=int(best_params['period']),
    buy_threshold=best_params['buy_threshold'],
    sell_threshold=best_params['sell_threshold']
)

result = kimsfinance_core.run_backtest(
    high=high, low=low, close=close, open_prices=open_p,
    volume=volume, timestamps=timestamps, strategy=optimal_strategy,
    initial_capital=10000.0, trading_fee=0.001,
    slippage=0.0005, use_gpu=False
)

from kimsfinance.visualization import print_performance_summary
print_performance_summary(result)

## Next Steps

For production genetic algorithms, consider:
- DEAP (Distributed Evolutionary Algorithms in Python)
- PyGAD (Python Genetic Algorithm)
- Multi-objective optimization (Sharpe + Drawdown)
- Island model for parallel evolution